# Task 45: Model Performance and Statistical Comparison

Compare scratch classifiers on shared stratified folds, summarize metric and bias-variance trade-offs, and estimate paired bootstrap intervals for F1 differences. The intervals are exploratory because models are selected from the same folds.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

candidates=[Path('heart.csv'),Path('Task_45_Model_Performance_Statistical_Comparison/heart.csv'),Path('Heart_Disease_Tasks/Task_45_Model_Performance_Statistical_Comparison/heart.csv')]
data=pd.read_csv(next(p for p in candidates if p.is_file())).drop_duplicates().reset_index(drop=True)
cols=[c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
X,y=data[cols].to_numpy(float),data.target.to_numpy(int)

def sigmoid(z):return 1/(1+np.exp(-np.clip(z,-35,35)))
class Logistic:
    def fit(self,x,y):
        self.w=np.zeros(x.shape[1]);self.b=0.
        for _ in range(900):
            p=sigmoid(x@self.w+self.b);e=p-y;self.w-=.06*(x.T@e/len(y)+.02*self.w);self.b-=.06*e.mean()
        return self
    def predict(self,x):return (sigmoid(x@self.w+self.b)>=.5).astype(int)
class Tree:
    def __init__(self,depth=4):self.depth=depth
    def fit(self,x,y):self.root=self.grow(x,y,0);return self
    def grow(self,x,y,d):
        count=np.bincount(y,minlength=2);node={'pred':int(np.argmax(count))}
        if d>=self.depth or len(y)<8 or len(np.unique(y))<2:return node
        parent=1-np.sum((count/len(y))**2);best=None
        for j in range(x.shape[1]):
            values=np.unique(x[:,j])
            for t in (values[:-1]+values[1:])/2:
                m=x[:,j]<=t
                if not m.any() or m.all():continue
                a=np.bincount(y[m],minlength=2)/m.sum();b=np.bincount(y[~m],minlength=2)/(~m).sum()
                gain=parent-m.mean()*(1-np.sum(a*a))-(~m).mean()*(1-np.sum(b*b))
                if best is None or gain>best[0]:best=(gain,j,t,m)
        if best is None or best[0]<=0:return node
        _,j,t,m=best;node.update(j=j,t=t,left=self.grow(x[m],y[m],d+1),right=self.grow(x[~m],y[~m],d+1));return node
    def predict(self,x):
        out=[]
        for row in x:
            node=self.root
            while 'j' in node:node=node['left'] if row[node['j']]<=node['t'] else node['right']
            out.append(node['pred'])
        return np.array(out)
class Forest:
    def fit(self,x,y,seed):
        rng=np.random.default_rng(seed);self.trees=[]
        for _ in range(6):
            ids=rng.integers(0,len(y),len(y));self.trees.append(Tree(4).fit(x[ids],y[ids]))
        return self
    def predict(self,x):return (np.mean([t.predict(x) for t in self.trees],axis=0)>=.5).astype(int)
class KNN:
    def fit(self,x,y):self.x,self.y=x,y;return self
    def predict(self,x):
        out=[]
        for row in x:
            ids=np.argsort(np.sum((self.x-row)**2,axis=1),kind='stable')[:7];out.append(int(self.y[ids].mean()>=.5))
        return np.array(out)

def metric(y_true,y_pred):
    tp=np.sum((y_true==1)&(y_pred==1));tn=np.sum((y_true==0)&(y_pred==0));fp=np.sum((y_true==0)&(y_pred==1));fn=np.sum((y_true==1)&(y_pred==0))
    p=tp/(tp+fp) if tp+fp else 0.;r=tp/(tp+fn) if tp+fn else 0.
    return {'accuracy':(tp+tn)/len(y_true),'precision':p,'recall':r,'f1':2*p*r/(p+r) if p+r else 0.}

def stratified_folds(labels,count=5,seed=45):
    rng=np.random.default_rng(seed);folds=[[] for _ in range(count)]
    for label in np.unique(labels):
        ids=np.flatnonzero(labels==label);rng.shuffle(ids)
        for i,part in enumerate(np.array_split(ids,count)):folds[i].extend(part.tolist())
    return [np.array(sorted(f)) for f in folds]

names=['Logistic regression','Decision tree','Random forest','k-NN'];folds=stratified_folds(y)
oof={name:np.zeros(len(y),dtype=int) for name in names};fold_rows={name:[] for name in names}
for fold,valid in enumerate(folds):
    valid_set=set(valid.tolist());train=np.array([i for i in range(len(y)) if i not in valid_set])
    xtr,xv=X[train],X[valid];ytr,yv=y[train],y[valid]
    mu,sd=xtr.mean(0),xtr.std(0);sd[sd==0]=1;xtr,xv=(xtr-mu)/sd,(xv-mu)/sd
    estimators=[Logistic().fit(xtr,ytr),Tree(4).fit(xtr,ytr),Forest().fit(xtr,ytr,100+fold),KNN().fit(xtr,ytr)]
    for name,model in zip(names,estimators):
        pred=model.predict(xv);oof[name][valid]=pred;fold_rows[name].append(metric(yv,pred))
summary=[]
for name in names:
    overall=metric(y,oof[name]);summary.append({'Model':name,**overall,'Fold F1 SD':np.std([m['f1'] for m in fold_rows[name]],ddof=1)})
performance=pd.DataFrame(summary).set_index('Model').sort_values('f1',ascending=False)
display(performance.round(3))

best=performance.index[0];rng=np.random.default_rng(145);comparisons=[]
for other in names:
    if other==best:continue
    diffs=[]
    class_ids=[np.flatnonzero(y==label) for label in np.unique(y)]
    for _ in range(1200):
        sample=np.concatenate([rng.choice(ids,len(ids),replace=True) for ids in class_ids])
        diffs.append(metric(y[sample],oof[best][sample])['f1']-metric(y[sample],oof[other][sample])['f1'])
    low,high=np.quantile(diffs,[.025,.975]);comparisons.append({'F1 comparison':f'{best} minus {other}','95% paired bootstrap interval':(round(low,3),round(high,3)),'CI excludes zero':bool(low>0 or high<0)})
display(pd.DataFrame(comparisons).set_index('F1 comparison'))
print('Bias-variance reading: a simple linear model may underfit interactions but is stable; deep trees can have low training bias and higher variance; forests average trees to reduce variance; k-NN has local flexibility and can be noisy at small k.')
print('Bootstrap intervals are exploratory after selecting the leader on these folds. Statistical confirmation requires a preselected comparison and independent external data.')